# Vector Database Data Ingestion 

In [1]:
import os
from pathlib import Path
from typing import Any, override
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader

load_dotenv(Path("../.env.local"))

from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

In [2]:
DATA_DIR = Path("../data")
COLLECTION_NAME = "documents"
QDRANT_URL = (os.getenv("QDRANT_URL") or "").strip()
QDRANT_API_KEY = (os.getenv("QDRANT_API_KEY") or "").strip() or None
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

if not QDRANT_URL:
    raise ValueError(
        "QDRANT_URL is not set. Run the cell above (load_dotenv) first, "
        "and ensure .env.local or .env contains QDRANT_URL=..."
    )

# Load Documents

In [3]:
loader = TextLoader(DATA_DIR / "data.txt")
documents = loader.load()

# Split documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = text_splitter.split_documents(documents)


In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
client = QdrantClient(url = QDRANT_URL, api_key = QDRANT_API_KEY)

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
    )

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

vector_store.add_documents(chunks)
print(f"Added {len(chunks)} documents to Qdrant")


Added 9 documents to Qdrant
